
### - DSL (Domain Specific Language) ->dropna(),withColumn() etc.
### - Learn how to Read the  data becomes data ingestion developer
### - Learn how to write the  data becomes Data egression developer
### - By Learning Transformation - DATA ENGINEER / DATA ANALYST /ETL DEVELOPER / DATA CURATION DEV


**Transformation we are going to achive in two ways , using DSL approach and using SQL approach**

## DATA Munging (data wrangling)
(Cleanup) Process of transforming and mapping data from Raw form into Tidy(usable) format with the intent of making it more appropriate and valuable for a variety of downstream purposes such for further Transformation/Enrichment, Egress/Outbound, analytics, Datascience/AI application & Reporting

**Types of Munging**
- Passive Data Munging - Data Discovery/Data Exploration/ EDA (Exploratory Data Analytics) (every layers ingestion/transformation/analytics/consumption) - Performing an (Data Exploration) exploratory data analysis of the raw data to identify the attributes and patterns.

- Active Data Munging
    Combining Data + Schema Evolution/Merging/Merging (Structuring)
    Validation, Cleansing, Scrubbing - Cleansing (removal of unwanted datasets), Scrubbing (convert raw to tidy)
    De Duplication and Levels of Standardization () of Data to make it in a usable format (Dataengineers/consumers)

**Difference b/w Active and Passive data munging (from chatGPT)**

**Active Data Munging **- Active data munging involves _**explicit actions performed by the user or developer**_ to clean or transform data.

**Examples:**

- Removing null values
- Converting data types
- Renaming columns
- Filtering rows
- Standardizing date formats
- Aggregating data

**Passive data munging** occurs when **_data preparation happens automatically_** through tools, frameworks, or predefined rules with minimal user intervention.

**Examples:**

- Automatic schema inference when reading CSV files
- Automatic type conversion by ETL tools
- Built-in data quality rules
- Metadata-driven transformations


passive Data Munging - EDA (Exploratory Data Analysis)

**manually understand the Data - manual EDA**
- header
- delimiter
- footer
- columns and datatypes
- comments
- record count
- duplicates / nulls / format issues

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv

## Programmatically perform EDA on source data

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv",inferSchema=True).toDF("custid","firstname","lastname","age","profession")

cust_df.show(10)
cust_df.printSchema()




In [0]:

print(type(cust_df))
print(cust_df.columns)  #returns column name from CSV

print(cust_df.dtypes)  #returns column name and datatype from CSV
print(cust_df.schema)  #returns schema structure of CSV file so we can assign it in variable and use it if we want but in printSchema (only print the schema)

In [0]:
cust_sample_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/cust_sample.csv")
cust_sample_df.show()

""" 
defaults:
    sep (delimiter)=,
    header=False
    inferSchema=False
"""


In [0]:
cust_sample_schema=cust_df.schema
cust_sample_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/cust_sample.csv",schema=cust_sample_schema)
cust_sample_df.show()

In [0]:
#find number of records in dataframe
print(cust_df.count())

#using distinct - unique records (row level)
#if the entire record is duplicate, we can remove it with distinct and dropDuplicates - both are same
# distinct() -record level duplication - distinct() - checks entire record row by row. if a particular record(row) matches the other row, it considered as duplicate

# dropDuplicates() without argument - record level duplication - similar like distinct
# dropDuplicates() with argument - column level duplication


print(cust_df.distinct().count())     #deduplication on record level

#whereas dropDuplicates() - one option provided by spark, if no arguments passed, it behaves like distinct() function -  checks entire record row by row. if a particular record(row) matches the other row, it considered as duplicate, returning the unique records

print(cust_df.dropDuplicates().count())    #deduplication on record level


#print(cust_df.distinct(["custid"]).count()) - gives error
# find column level unique count

cust_df2 = cust_df.select("custid")

cust_df2.printSchema()
cust_df2.show(5)


print(cust_df.select("custid").distinct().count()) # find column level unique count 
#when we go with select(custid).distinct(), it will return unique custid alone


#case2, need all columns with uniqueness based on custid
#remove duplicates based on custid
print(cust_df.dropDuplicates(["custid"]).count()) # deduplicated on column level and return the entire datframe 

# Deduplication is the process of identifying and removing duplicate records from a dataset to ensure data quality and uniqueness. In PySpark, it is commonly done using distinct() or dropDuplicates().


to create a new data frame with deduplicated data

new_df=cust_df.dropDuplicates()

In [0]:
#print unique custid column
#return only custid
cust_df.select("custid").distinct().show(5)

#column level duplication possible only using dropDuplicates()
#remove duplicates based on custid
#return all the column after deduplication based on custid
cust_df.dropDuplicates(["custid"]).show((5))


**describe()** provides basic descriptive statistics for numeric and string columns.

Statistics Returned
- count
- mean
- stddev
- min
- max

In [0]:
display(cust_df.describe())


**summary()** is more flexible and provides additional statistics.

- count
- mean
- stddev
- min
- 25%
- 50% (median)
- 75%
- max

In [0]:
display(cust_df.summary())

cust_df.summary().show()

In [0]:
cust_df.filter("custid is null").show()

# **scenarios to create Dataframe**

**suppose i have customer data in diffrent files in same dir - read dir**

**suppose i have customer data in diffrent files in same dir with sub dir as well - read main dir with recursive_lookup enable**


**suppose i have customer data and sales data in diffrent files in same dir , i want to read only sales data - read dir with file pattern (/data/sales*)**


** suppose i have customer data and sales data in diffrent files in same dir and sub dir , i want to read only sales data - read main dir with recursivefilelookup and pathGlobfilter="sales*" **

**suppose i have sales data in diff directories - list of path or list of files**

**Schema evolution**  changes in the scehma

Day 1 to Day 5 files have cid ,cname ,age

Day 5 to day 10 file have cid ,cname ,age , profession

Day 11 - cid , cname , mobile , profession

we achived this writing into some columnar parquet / orc file format

while reading the entire data we will use with mergeschema option

combining Data -> schema Evolution / Structuring

In [0]:
parquet_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day3.csv",header=True,inferSchema=True)
parquet_df.show()
parquet_df.printSchema()
parquet_df.write.mode("append").parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_write")


In [0]:
parquet_read_df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_write")
parquet_read_df.show()
parquet_read_df.printSchema()

In [0]:
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/",header=True,inferSchema=True)
student_df.show()
student_df.printSchema()

In [0]:
student_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv",header=True,inferSchema=True)
student_df1.show()
student_df1.printSchema()

In [0]:
student_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv",header=True,inferSchema=True)
student_df2.show()
student_df2.printSchema()

#since union can check only the number of columns and its datatypes
#both condition satisfied so provided the union list]
#in spark, union does not remove the duplicates
student_df1_df2=student_df1.union(student_df2)
student_df1_df2.show()
student_df1_df2.count()

In [0]:
student_df3=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)
student_df3.show()
student_df3.printSchema()


student_df_4=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",header=True,inferSchema=True)
student_df_4.show()
student_df_4.printSchema()


# union will work when we have dtafrmes with same number of columns and datatype so it gives error
#error : The value 'chennai' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type

student_df1_df3=student_df1.union(student_df3)
student_df1_df3.show()
student_df1_df3.count()

In [0]:
student_df_5=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv",header=True,inferSchema=True)
student_df_5.show()
student_df_5.printSchema()

#again number of columns mismatch so it gives error
#error :but the first input has 4 columns and the second input has 3 columns
student_df_5_df1=student_df_5.union(student_df1)
student_df_5_df1.show()
student_df_5_df1.count()


In [0]:
# SQL union

# union will work on same number of columns and datatypes based on positions
# usually in sql, union will return the unique records

#in spark DS, union will allow duplicates
#unionby name will consolidate all column with allowmissingcolumns=True option


data1=[(100,"raja",23),(102,"ram",25),(103,"ravi",27),(503,"madhu",45)]
data2=[(500,"rajesh",29),(501,"suresh",30),(502,"rajesh",43),(503,"madhu",45)]
data3=[(500,"rajesh",2000),(501,"suresh",2001),(502,"rajesh",2002),(503,"madhu",2003)]
sql_df1=spark.createDataFrame(data1,schema=["id","name","age"])
sql_df2=spark.createDataFrame(data2,schema=["id","name","age"])
sql_df3=spark.createDataFrame(data3,schema=["id","name","year"])

sql_df1.show()
sql_df2.show()

#combine both df into one
combined_df=sql_df1.union(sql_df2)
combined_df.show()
combined_df.count()


print("id, name, age with ID,NAME and Year")
sql_df3.show()
combined_df1=sql_df1.union(sql_df3)
combined_df1.show()
combined_df1.count()

data4=[(100,23,"raja"),(102,25,"ram")]
sql_df4=spark.createDataFrame(data4,schema=["id","age","name"])
sql_df4.show()



#union of id","name","age"with "id","age","name"
print("id, name, age with id,age,name")
combined_df2=sql_df1.unionByName(sql_df4)
combined_df2.show()
combined_df2.count()


#union of id","name","age"with "id","name","year"
print("id, name, age with id,age,name")
combined_df2=sql_df1.unionByName(sql_df3,allowMissingColumns=True)
combined_df2.show()
combined_df2.count()




**Data munging** (also called **data wrangling**) is the process of transforming and cleaning raw data into a format that can be easily used for analysis, reporting, or machine learning.

# 2. Validation, cleansing and scrubbing

handling missing values, handling mull values

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified


# clean up data which ever not matching with schema 

# reject process - 1

In [0]:
schema_structure="id int,fname string,lname string,age int,profession string,corrupt_record string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE",columnNameOfCorruptRecord="corrupt_record")
read_df.show()
read_df.printSchema()


error_df=read_df.filter("corrupt_record is not null")
error_df.show(10)  # 5 records

valid_df=read_df.filter("corrupt_record is null")
valid_df.show(10)  #10000 records


#error_df.cache()
error_df.select("corrupt_record").show()


#how to write the error record into separate table
error_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_csv")




In [0]:


# 2 level cleansing / rejection 

# cleaning up / drop the records 

#  null handling - null record removal 

# null - single column or multiple columns may have null , entire rec may have null 

# single null - remove that recod -> delete rec when col is null 
# mulit col null - remove that recod -> delete rec when col is null and col2 is null
# schema_structure="cust_id int,fname string,lname string,age int,profession string"
schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")

# handle null -> spark DSL  -> na functions 
# na -> not applicable -> null
read_df.show(20)
print(read_df.count())


# na.drop -> have 3 arg -> subset , how , threshold
#  subset -> default is all columns -> option as list of columns 
#  how ->default is  any -> options are  ->   all | any 
#               any -> any one column is the subset is null -> remove that record ->  or
#               all -> all columns are null is the subset  -> remove that record -> and




# if all columns are null remove that record 
print("if all columns are null")
null_record_df=read_df.na.drop(how="all")
null_record_df.show()
print(null_record_df.count())


print("if any columns have null, remove that record")
# if any one columns have null, remove that record
null_record_df=read_df.na.drop(how="any")
null_record_df.show()
print(null_record_df.count())


#custid is the key and it should not have null values so if custid is null, then remove that record
print("if id is the key")
cust_id_null_record_df=read_df.na.drop(subset="cust_id",how="any")
cust_id_null_record_df.show()
print(cust_id_null_record_df.count())



print("custid and age both are null")
#custid and age is the key and it should not have null values so if custid and age both are null, then remove that record
cust_id_null_record_df=read_df.na.drop(subset=["cust_id","age"],how="all")
cust_id_null_record_df.show()
print(cust_id_null_record_df.count())


print("custid or age have null")
#custid or age  have null values, then remove that record
cust_id_null_record_df=read_df.na.drop(subset=["cust_id","age"],how="any")
cust_id_null_record_df.show()
print(cust_id_null_record_df.count())


In [0]:

schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")
filter_read_df=read_df.filter("cust_id IS NULL OR age IS  NULL")
filter_read_df.show()
filter_read_df.count()










In [0]:
# na.fill -> 3 arg -> subset , value , inplace 
# scrubbing 
# nvl , coalesce 


not_null_df=read_df.na.fill(0,subset=["age"]).na.fill("N/A",subset=["fname","lname"])
not_null_df.show()
not_null_df.count()


In [0]:
#drop - na.drop() or dropna()

schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")
read_df.show(10)
print(read_df.count())

#na.drop()=dropna()
#defaults: subset=all, how=any

#using na.drop
#na.drop(subset,how,threshold)
null_dropped=read_df.na.drop()
null_dropped.show(5)
print(null_dropped.count())

#using dropna
null_droppedna=read_df.dropna()
null_droppedna.show(5)
null_droppedna.count()


#threshold -int
#thresh=2 ->atleast 2 columns should not be null
null_dropped_thresh=read_df.na.drop(thresh=2)
null_dropped_thresh.show(5)
print(null_dropped_thresh.count())


#threshold -int
#thresh=2 ->atleast 2 columns should not be null (any 2 columns have column, it will not drop)
null_dropped_thresh=read_df.na.drop(thresh=5)
null_dropped_thresh.show(5)
print(null_dropped_thresh.count())

In [0]:
data=[(1,"James",None,"36636","M",3000),
      (2,"Rose",None,"40288",None,4000),
      (3,None,None,"42114","M",None),
      (4,"Maria","Jones","39192","F",4000),
      (None,"Jen","Mary","Brown",None,-1)
     ]
df=spark.createDataFrame(data,["id","fname","lname","ssn","gender","salary"])
df.show()
df.count()

#default - so any column have null, it will drop that record
drop_df=df.dropna()
drop_df.show()
drop_df.count()


#using all,so if all columns have null, it will drop
drop_df=df.dropna(how="all")
drop_df.show()
drop_df.count()


#using subset - any of these 2 subset have null, it will remove
drop_df=df.dropna(subset=["fname","lname"],how="any")
drop_df.show()
drop_df.count()


#using subset,thresh=1 - any of these 2 subset have value, it will hold the record
drop_df=df.dropna(subset=["fname","lname"],how="any",thresh=1)
drop_df.show()
drop_df.count()


In [0]:
# fill - fillna() or na.fill()


data=[(1,"James",None,"36636","M",3000),
      (2,"Rose",None,"40288",None,4000),
      (3,None,None,"42114","M",None),
      (4,"Maria","Jones","39192","F",4000),
      (None,"Jen","Mary","Brown",None,-1)
     ]
df=spark.createDataFrame(data,["id","fname","lname","ssn","gender","salary"])

df.show()
df.printSchema()
print(df.count())



# na.fill()= fillna()
# for integer,decimal 0 and string "NA"
# date and timestamp column, fill will not work

#assign/ fill 0 value to all integer column which have NULL
nonull_df=df.na.fill(0)
nonull_df.show()

#assign/ fill 0 value to specific id column which have NULL
df.na.fill(0,subset="id").show()



#assign/ fill 0 value to all integer column which have NULL
df.na.fill(0,subset="id").fillna("unknown",subset="fname").show()


#assign/ fill 0 value to all integer column which have NULL
df.na.fill(0,subset="id").fillna("unknown",subset=["fname","lname"]).show()


#assing 0 for integer and unknown for string datatype which have NULL
df.na.fill(0).na.fill("NA").show()

In [0]:
#replace - it will replace everywhere in the dataset by default
#if you want to limit the replace to particular column, use subset
#to specify more values, use dictionary or list
#replace (oldvalue,newvalue,subset)
df.show()
df.count()

#for one value 
replace_df=df.replace("James","John")
replace_df.show()

#for multiple values use dictionary,list
replace_df=df.replace({"Jones":"arjun","Rose":"Mary"})
replace_df.show()

#for multiple values use lsit
replace_df=df.replace(["M","F"],["Male","Female"])
replace_df.show()


#for multiple values use dictionary
replace_df=df.replace({3000:30000,-1:50000,4000:40000},subset="salary")
replace_df.show()

# **3. Standarisation** 

**Making the data more standard by adding,removing,reordering column as per the expected standard, unifying into expected format, converting the type as expected etc**

In [0]:

# additional audit columns 
# load_date, system ,mod_date, user,applicatioid

# how to add columns with staic value in pyspark  
#   -- using withColumn

# SQL -> select 'customer' as system,current_date as load_date , * from staging_cust

# DSL (domain specific language) --> select , withColumn
# withColumn(new_col_name,value ) 
# value -> should be in the column format
# select *,'customerdata' as source from tbl

# withcolumn -> hardcoded using lit 
#            -> taking another column using col
#            -> using built in sql function which return column

read_df.columns

#lit = create a value for the new column 
#col = if already a column in dataframe, just want to create new column and copy the values from existing column
from pyspark.sql.functions import lit,col,current_date

#lit = create a value for the new column 
#col = if already a column in dataframe, just want to create new column and copy the values from existing column
from pyspark.sql.functions import lit,col
#how to add a column to datasource using withColumn in DSL


#ERROR: Argument `col` should be a Column, got str.
#read_enrichment_df=read_df.withColumn("Source","Customerdata")  #- error because withcolumn value cannot be string


#for new value
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata"))
read_enrichment_df.show(5)

#for existing value in df and copy the value as a new column
read_enrichment_df=read_df.withColumn("Dupli_Profession",col("profession"))
read_enrichment_df.show(5)


#for doing multiple columns
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")) 
read_enrichment_df.show()
read_enrichment_df.printSchema()

#to load currentDate() as new column
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")).withColumn("Load_Date",current_date())
read_enrichment_df.show()
read_enrichment_df.printSchema()


#how to add a column to datasource using withColumn in DSL


#ERROR: Argument `col` should be a Column, got str.
#read_enrichment_df=read_df.withColumn("Source","Customerdata")  #- error because withcolumn value cannot be string


#for new value
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata"))
read_enrichment_df.show(5)

#for existing value in df and copy the value as a new column
read_enrichment_df=read_df.withColumn("Dupli_Profession",col("profession"))
read_enrichment_df.show(5)


#for doing multiple columns
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")) 
read_enrichment_df.show()
read_enrichment_df.printSchema()

#to load currentDate() as new column
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")).withColumn("Load_Date",current_date())
read_enrichment_df.show()
read_enrichment_df.printSchema()



**# Check the below one -- June 18**

In [0]:
#profesion wise count using groupby

read_profession_df=read_enrichment_df.groupBy("profession").count()
read_profession_df.show(50,False)


#NULL have 90 records
#Pilot has 210 records

#i want to update NULL to Pilot
read_profession_df.filter(col("profession").isNull()).show()
update_pilot_df=read_profession_df.replace("NULL","PILOT",subset=["profession"])
update_pilot_df.show(100,False)



In [0]:

from pyspark.sql.functions import lit, col, current_date

df=spark.range(20)
df.show(5)
df.printSchema()
print(df.schema)


#create 3 more columns
#create new column called id2= id column *2 (using col)
#Create load_dt as current_date (using builtin funciton)
#create createdby as dbuser (using lit)
#equivalent sql - select id2 as id*2, current_date() as load_dt, lit("dbuser") as createdby

df2=df.withColumn("id2",col("id")*2).withColumn("load_dt",current_date()).withColumn("createdby",lit("dbuser"))
df2.show(5)
df2.printSchema()



import getpass
current_user = getpass.getuser()
#create new column as current_user

print("Updated dataframe")
df3=df2.withColumn("CurrentUser",lit(current_user))
df3.show(5,False)





In [0]:
#using withcolumn we added columns
#now using select

from pyspark.sql.functions import *
df=spark.range(20)
df.show(5)
df.printSchema()
print(df.schema)

df2=df.select("id",(col("id")*2).alias("id2"),current_date().alias("load_dt"),lit("dbuser").alias("createdby"))
df2.show(5)
df2.printSchema()

#using select we can add columns
#using withcolumn we can add columns and rename columns
#)
df2.show(5)
df2.printSchema()





In [0]:
schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")
read_df.show(5)
read_df.select("*").show(5)
read_df.select("fname","lname").show(5)
read_df2=read_df.withColumn("fullname",concat(col("fname"),lit(" "),col("lname")))
read_df2.show(5)

#using select
read_df3=read_df.select("fname","lname",concat(col("fname"),lit(" "),col("lname")).alias("fullname"))
read_df3.show(5)
read_df3.printSchema()




# **Standarization -2 uniformality**

In [0]:
read_df.show(5)

from pyspark.sql.functions import upper
#profession, receiving in multiple cases, lower or upper,initcap

standard_df=read_df.withColumn("profession",upper(col("profession")))
standard_df.show(5)
standard_df.printSchema()

standard_df=read_df.withColumn("Updated_Prof",upper(col("profession")))
standard_df.show(5)
standard_df.printSchema()


standard_df2=read_df.select("profession",upper(col("profession")).alias("Updated_Prof"))
standard_df2.show(5)
standard_df2.printSchema()

standard_df3=read_df.select("profession",upper(col("profession")).alias("Updated_Prof"),concat(col("fname"),lit(" "),col("lname")).alias("FullName"))
standard_df3.show(5)
standard_df3.printSchema()





# **Standarisation 3 - typecasting**

 

In [0]:
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",mode="PERMISSIVE").toDF("cust_id","fname","lname","age","profession")
read_df.show(5)
read_df.printSchema()

#sql - select cast(age as int) as age from cust_df

from pyspark.sql.functions import col
#filter out the number that causes the error and do the type conversion
read_df2=read_df.filter("age!='7-7'").withColumn("age",col("age").cast("int"))
read_df2.show(5)
read_df2.printSchema()


#using regular expression for non standard record (to get integer - ^[0-9]+$)

read_df.filter("age rlike '^[0-9]+$'").show()
read_df.filter("age not rlike '^[0-9]+$'").show()

read_df2=read_df.filter("age rlike '^[0-9]+$'").withColumn("age",col("age").cast("int"))
read_df2.show(5)
read_df2.printSchema()



# Standarisation - 4 

#Naming and Reordering

In [0]:
read_df.show(5)

 #rename the column

#using withColumnRenamed options
read_df2=read_df.withColumnRenamed("fname","firstName").withColumnRenamed("lname","lastName")
read_df2.show(5)
read_df2.printSchema()


#using select option
read_df2=read_df.select("cust_id",col("fname").alias("firstName"),col("lname").alias("lastName"),"age","profession")
read_df2.show(5)


In [0]:
#remove a column
#select only required column
#drop
from pyspark.sql.functions import *


#using withcolumn
read_df1=read_df.withColumn("fullname",concat(col("fname"),lit(" "),col("lname")))
read_df1.show(5)
read_df1.printSchema()

#now i want to drop the fname and lname column
read_df2=read_df1.drop("fname","lname")
read_df2.show(5)

#using select rearranging the order i want - custid,name,age,prof
read_df3=read_df2.select("cust_id","fullname","age","profession")
read_df3.show(5)



In [0]:

#input_file_name is a spark fnction but not working in databricks but it so use _metadata

filename = "cust_info_south_20260618.csv"

base = filename.replace(".csv", "")
parts = base.split("_")

print(parts)
source = parts[-2]
data_dt = parts[-1]

print(source)   # south
print(data_dt)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/cust_data/emp_NAC_20260619.csv

In [0]:
#input_file_name is a spark fnction but not working in databricks but it so use _metadata

from pyspark.sql.functions import input_file_name,col,lit,split,current_user,concat

#read csv file

cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/cust_data").toDF("cust_id","fname","lname","age","profession")



#to get the file name from directory data using(input_file_name but its not working in databricks so we use _metadata)

#Error : The command(s): input_file_name are not supported in Unity Catalog. Please use _metadata.file_path instead. SQLSTATE: 0AKUC
cust_df_metadata=cust_df.withColumn("filename",col("_metadata"))
cust_df_metadata.printSchema()

#taking filename alone from metadata
cust_df_filename=cust_df_metadata.withColumn("filename",col("_metadata.file_name"))

#removing .csv from filename using split option
cust_df_filename=cust_df_metadata.withColumn("filename",split(col("_metadata.file_name"),"\\.")[0])

#split the filename for source and datadate column
cust_df_source_datadt_df=cust_df_filename.withColumn("source",split(col("filename"),"_")[1]).withColumn("data_dt",split(col("filename"),"_")[2]).drop("filename")


#adding createdby Column with createduser
cust_df_createduser=cust_df_source_datadt_df.withColumn("createdby",current_user())


#add fname and lname and make it a fullname
cust_df_fullname=cust_df_createduser.withColumn("fullname",concat(col("fname"),lit(" "),col("lname"))).drop("fname","lname")

cust_df_final=cust_df_fullname.select("cust_id","fullname","age","profession","source","data_dt","createdby")
cust_df_final.show(10)
cust_df_final.printSchema()

cust_df_final.write.mode("overwrite").saveAsTable("izwd37dev.wd37db.cust_info_table")


#cust_df_source_datadt_df.write.mode("overwrite").saveAsTable("cust_info_silver")









In [0]:
%sql

select(*) from izwd37dev.wd37db.cust_info_table

# Select Vs SelectExpr

select - is for pyspark dsl col function
selectExpr - for pyspark sql expression



# withColumn Vs withColumns

In [0]:
#withColumn - adding one column
#withColumns - adding multiple columns using dictionary (key,value pairs)

